In [0]:
from pyspark.sql.functions import col, from_json, when, to_timestamp
from pyspark.sql.types import *

schema = StructType([
    StructField("sensor_id", StringType()),
    StructField("sensor_timestamp", TimestampType()),
    StructField("value", DoubleType()),
    StructField("unit", StringType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("district", StringType()),
    StructField("topic", StringType())
])

bronze = spark.table("workspace4sadt.bronze.sensors_raw")

parsed = bronze.withColumn(
    "json", from_json(col("value"), schema)
).select(
    "json.*",
    "ingest_time"
)

silver = parsed \
    .withColumn(
        "is_valid",
        col("latitude").between(-90, 90) &
        col("longitude").between(-180, 180) &
        col("value").isNotNull()
    ) \
    .withColumn(
        "value_si",
        when(col("unit") == "km/h", col("value") * 1000 / 3600)
        .otherwise(col("value"))
    ) \
    .withColumn(
        "alert_flag",
        col("value") > 80
    ) \
    .dropDuplicates(["sensor_id", "sensor_timestamp"])

silver.write.format("delta") \
    .mode("append") \
    .saveAsTable("workspace4sadt.silver.sensors_cleaned")


In [0]:
%sql
-- SELECT * FROM workspace4sadt.silver.sensors_cleaned;

sensor_id,sensor_timestamp,value,unit,value_si,longitude,latitude,district,topic,ingest_time,is_valid,alert_flag
transport-7,2025-12-19T18:29:22.361Z,99.64171931790041,passengers,99.64171931790041,23.725778520305933,38.02543625505692,5,transport.sensors,2025-12-19T18:29:23.284Z,true,true
environment-40,2025-12-19T18:29:22.361Z,12.67506570056267,ppm,12.67506570056267,23.74171171896385,38.002716718953735,2,environment.sensors,2025-12-19T18:29:23.284Z,true,false
traffic-7,2025-12-19T18:29:22.361Z,76.39964741104912,km/h,21.222124280846977,23.71604169204078,38.02233303297395,1,traffic.sensors,2025-12-19T18:29:23.284Z,true,false
environment-37,2025-12-19T18:29:45.995Z,13.163228140080985,ppm,13.163228140080985,23.74047255718388,37.99420299929129,5,environment.sensors,2025-12-19T18:29:46.410Z,true,false
greeninfra-11,2025-12-19T18:30:09.756Z,93.99742297572303,liters,93.99742297572303,23.73968390585893,37.984800952646765,1,greeninfra.sensors,2025-12-19T18:30:10.152Z,true,true
environment-9,2025-12-19T18:29:22.361Z,69.91658334773838,ppm,69.91658334773838,23.73362451802759,38.01039180839339,5,environment.sensors,2025-12-19T18:29:23.284Z,true,false
environment-44,2025-12-19T18:30:09.756Z,17.87120937342766,ppm,17.87120937342766,23.720647386986876,38.02804793197668,5,environment.sensors,2025-12-19T18:30:10.152Z,true,false
greeninfra-38,2025-12-19T18:29:45.995Z,58.013405098083325,liters,58.013405098083325,23.738937776124267,38.00060616455941,4,greeninfra.sensors,2025-12-19T18:29:46.410Z,true,false
greeninfra-18,2025-12-19T18:29:42.294Z,57.68796861166051,liters,57.68796861166051,23.707864147753913,38.00730365082389,5,greeninfra.sensors,2025-12-19T18:29:42.717Z,true,false
transport-4,2025-12-19T18:29:22.361Z,90.2495495880455,passengers,90.2495495880455,23.72362648454599,37.98915518702448,1,transport.sensors,2025-12-19T18:29:23.284Z,true,true
